In [17]:
import numpy as np
import pandas as pd
import cv2
from matplotlib import pyplot as plt
from numpy.linalg import inv
import os

from colormath.color_conversions import convert_color
from colormath.color_objects import LabColor, XYZColor
from colormath.color_diff import delta_e_cie2000

from skimage.color import rgb2lab, lab2rgb

from scipy.interpolate import BSpline, make_interp_spline

from scipy.optimize import minimize
from scipy.optimize import dual_annealing
from scipy.optimize import least_squares
from scipy.optimize import basinhopping

from numpy.random import random
from numpy.random import rand

import skimage.measure
from skimage.metrics import structural_similarity as ssim

from sklearn.metrics import mean_squared_error
import math

from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# Load Image and Evaluate(Entropy, SSIM, rmse)

In [7]:
def cal_entropy1(img):
    return skimage.measure.shannon_entropy(img)

def cal_ssim(imga, imgb):
    graya = cv2.cvtColor(imga, cv2.COLOR_BGR2GRAY)
    grayb = cv2.cvtColor(imgb, cv2.COLOR_BGR2GRAY)

    (score, diff) = ssim(graya, grayb, full=True)
    
    return score

def cal_rmse(imga, imgb):
    graya = cv2.cvtColor(imga, cv2.COLOR_BGR2GRAY)
    grayb = cv2.cvtColor(imgb, cv2.COLOR_BGR2GRAY)

    MSE = mean_squared_error(graya, grayb)
    RMSE = math.sqrt(MSE)
    
    return RMSE

In [8]:
base_path = 'DT02'
dump_file = 'nbi_res_2.csv'

ent_wli_list = []
ent_nbi1_list = []
ent_nbi2_list = []
ent_nbi3_list = []

ssim_nbi1_list = []
ssim_nbi2_list = []
ssim_nbi3_list = []

rmse_nbi1_list = []
rmse_nbi2_list = []
rmse_nbi3_list = []

for i in range(20):
    tar_id = i + 1
    
    wli_file = base_path + '/WLI/%02d.jpg'%(tar_id)
    nbi1_file = base_path + '/nbi_sim/%02d_light%d.jpg'%(tar_id, 1)
    nbi2_file = base_path + '/nbi_sim/%02d_light%d.jpg'%(tar_id, 2)
    nbi3_file = base_path + '/nbi_sim/%02d_light%d.jpg'%(tar_id, 3)
     
    wli = cv2.imread(wli_file)
    nbi1 = cv2.imread(nbi1_file)
    nbi2 = cv2.imread(nbi2_file)
    nbi3 = cv2.imread(nbi3_file)
    
    ent_wli = cal_entropy1(wli)
    ent_nbi1 = cal_entropy1(nbi1)
    ent_nbi2 = cal_entropy1(nbi2)
    ent_nbi3 = cal_entropy1(nbi3)
    
    ent_wli_list.append(ent_wli)
    ent_nbi1_list.append(ent_nbi1)
    ent_nbi2_list.append(ent_nbi2)
    ent_nbi3_list.append(ent_nbi3)
    
    ssim_nbi1 = cal_ssim(wli, nbi1)
    ssim_nbi2 = cal_ssim(wli, nbi2)
    ssim_nbi3 = cal_ssim(wli, nbi3)

    rmse_nbi1 = cal_rmse(wli, nbi1)
    rmse_nbi2 = cal_rmse(wli, nbi2)
    rmse_nbi3 = cal_rmse(wli, nbi3)
    
    ssim_nbi1_list.append(ssim_nbi1)
    ssim_nbi2_list.append(ssim_nbi2)
    ssim_nbi3_list.append(ssim_nbi3)
    
    rmse_nbi1_list.append(rmse_nbi1)
    rmse_nbi2_list.append(rmse_nbi2)
    rmse_nbi3_list.append(rmse_nbi3)

In [9]:
df = pd.DataFrame()

df['ent_wli'] = ent_wli_list
df['ent_nbi1'] = ent_nbi1_list
df['ent_nbi2'] = ent_nbi2_list
df['ent_nbi3'] = ent_nbi3_list

df['ssim_nbi1'] = ssim_nbi1_list
df['ssim_nbi2'] = ssim_nbi2_list
df['ssim_nbi3'] = ssim_nbi3_list

df['rmse_nbi1'] = rmse_nbi1_list
df['rmse_nbi2'] = rmse_nbi2_list
df['rmse_nbi3'] = rmse_nbi3_list

df

,ent_wli,ent_nbi1,ent_nbi2,ent_nbi3,ssim_nbi1,ssim_nbi2,ssim_nbi3,rmse_nbi1,rmse_nbi2,rmse_nbi3
0,7.041621,7.084143,7.084143,7.084143,0.885675,0.885675,0.885675,10.475012,10.475012,10.475012
1,7.145020,7.168508,7.168508,7.168508,0.893545,0.893545,0.893545,10.713931,10.713931,10.713931
2,7.377287,7.262888,7.262888,7.262888,0.867624,0.867624,0.867624,10.318620,10.318620,10.318620
3,7.359484,7.343422,7.343422,7.343422,0.920433,0.920433,0.920433,11.200500,11.200500,11.200500
4,7.307207,7.402809,7.402809,7.402809,0.892552,0.892552,0.892552,10.914701,10.914701,10.914701
5,7.253523,7.097246,7.097246,7.097246,0.911113,0.911113,0.911113,11.058016,11.058016,11.058016
6,6.882287,6.905066,6.905066,6.905066,0.880564,0.880564,0.880564,9.993728,9.993728,9.993728
7,7.257917,7.210631,7.210631,7.210631,0.872390,0.872390,0.872390,10.297629,10.297629,10.297629
8,6.950633,7.193985,7.193985,7.193985,0.848487,0.848487,0.848487,10.806254,10.806254,10.806254
9,7.040947,7.261015,7.261015,7.261015,0.876053,0.876053,0.876053,10.207433,10.207433,10.207433


In [10]:
df.to_csv(dump_file)

In [42]:
nbis = np.load('data_nbi.npy')

caps1 = np.load('data_cap1.npy')
caps2 = np.load('data_cap3.npy')
caps3 = np.load('data_cap5.npy')

In [43]:
md_path = 'nbi'
model = keras.models.load_model(md_path)

model.load_weights('cap5.hdf5')

nbi_weights = model.get_weights() 

In [44]:
def to_nbi(img):
    
    img_shape = img.shape
    
    tar_img = img.reshape(-1,3) / 255
    
    tar_img = np.dot(tar_img, nbi_weights[0]) + nbi_weights[1]
    tar_img = np.dot(tar_img, nbi_weights[2]) + nbi_weights[3]
    tar_img = np.dot(tar_img, nbi_weights[4]) + nbi_weights[5]

    tar_img = tar_img * 255

    tar_img = tar_img.reshape(img_shape)
    
    tar_img[tar_img>255] = 255
    tar_img[tar_img<0] = 0
    
    tar_img = tar_img.astype('uint8')
    
    return tar_img

In [45]:
from_color = nbis

# from_color = from_color * 1.2

# from_color[0:18, 0] = from_color[0:18, 0] + 35
# from_color[0:18, 1] = from_color[0:18, 1] - 20
# from_color[0:18, 2] = from_color[0:18, 2] - 22

# from_color[from_color>255] = 255
# from_color[from_color<0] = 0

# from_color = from_color.astype('uint8')

to_color = to_nbi(caps3)

In [46]:
def cal_delta_e_cie2000_lab(lab_src, lab_test):
    
    errs = np.zeros(lab_src.shape[0])
    
    for i in range(lab_src.shape[0]):
        
        color0 = LabColor(lab_l=lab_src[i,0], lab_a=lab_src[i,1], lab_b=lab_src[i,2])
        color1 = LabColor(lab_l=lab_test[i,0], lab_a=lab_test[i,1], lab_b=lab_test[i,2])

        errs[i] = delta_e_cie2000(color0, color1)
        
    return errs

def cal_delta_e_cie2000_rgb(rgb_src, rgb_test):
    
    lab_src = rgb2lab(rgb_src / 255.0)
    lab_test = rgb2lab(rgb_test / 255.0)
    
    errs = np.zeros(lab_src.shape[0])
    
    for i in range(lab_src.shape[0]):
        
        color0 = LabColor(lab_l=lab_src[i,0], lab_a=lab_src[i,1], lab_b=lab_src[i,2])
        color1 = LabColor(lab_l=lab_test[i,0], lab_a=lab_test[i,1], lab_b=lab_test[i,2])

        errs[i] = delta_e_cie2000(color0, color1)
    
    # ignore gray
#     return errs[:-6]

    return errs

def rmse(errs):
    return np.sqrt(((errs) ** 2).mean())

In [47]:
errs = cal_delta_e_cie2000_rgb(from_color, to_color)

In [48]:
df = pd.DataFrame()

df['F_R'] = from_color[:,0]
df['F_G'] = from_color[:,1]
df['F_B'] = from_color[:,2]

df['T_R'] = to_color[:,0]
df['T_G'] = to_color[:,1]
df['T_B'] = to_color[:,2]

df['CIEDE2000'] = errs
df

,F_R,F_G,F_B,T_R,T_G,T_B,CIEDE2000
0,67,71,58,72,76,61,1.851349
1,119,123,101,119,128,106,2.667975
2,98,138,115,96,142,118,1.702441
3,90,70,57,88,65,52,2.083365
4,115,167,143,103,159,133,2.945361
5,155,147,123,149,149,125,3.954737
6,96,65,54,112,68,55,4.594357
7,92,160,136,76,164,137,2.563585
8,80,113,92,79,108,89,2.008910
9,61,115,94,53,97,80,6.315892


In [49]:
df.to_csv('CIEDE2000.csv')